# Lab Assignment 1 — Gaussian Naïve Bayes on the Iris Dataset

**Name:** Ghanshyam Ghimire  
**University:** Kathmandu University — BTech in Artificial Intelligence, 4th Semester  
**Subject:** Introduction to Machine Learning  
**Instructor:** Sandeep Gupta  

---

## Objectives

1. Implement **Gaussian Naïve Bayes from scratch** — no `sklearn.naive_bayes`.
2. Print predicted vs. actual labels on the test set.
3. Compute the **confusion matrix** (3×3) and report **accuracy, precision, recall, F1** (per-class and macro-averaged).

## Dataset

The Iris dataset contains 150 samples across three classes — **setosa**, **versicolor**, **virginica** —  
each described by four features: sepal length, sepal width, petal length, petal width (all in cm).

> **Note:** `sklearn` is used **only** for loading the dataset and splitting. The classifier itself is built entirely from scratch using NumPy.

---

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load Iris (sklearn used only for data loading, NOT for the classifier)
iris          = load_iris()
X             = iris.data            # shape (150, 4)
y             = iris.target          # 0=setosa, 1=versicolor, 2=virginica
class_names   = iris.target_names    # array(['setosa', 'versicolor', 'virginica'])
feature_names = iris.feature_names

# Reproducible 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y        # preserves class proportions
)

print('=' * 44)
print('  Iris Dataset — Summary')
print('=' * 44)
print(f'  Total samples   : {X.shape[0]}')
print(f'  Features        : {X.shape[1]}')
print(f'  Classes         : {list(class_names)}')
print(f'  Training set    : {X_train.shape[0]} samples')
print(f'  Test set        : {X_test.shape[0]} samples')
print('=' * 44)
print()
print('Feature names:')
for i, fn in enumerate(feature_names):
    print(f'  [{i}] {fn}')

  Iris Dataset — Summary
  Total samples   : 150
  Features        : 4
  Classes         : ['setosa', 'versicolor', 'virginica']
  Training set    : 120 samples
  Test set        : 30 samples

Feature names:
  [0] sepal length (cm)
  [1] sepal width (cm)
  [2] petal length (cm)
  [3] petal width (cm)


In [2]:
class GaussianNaiveBayes:
    """
    Gaussian Naïve Bayes Classifier — implemented from scratch.

    Training  : compute per-class mean, variance, and log-prior.
    Inference : argmax over log P(y) + sum of log Gaussian likelihoods.
    """

    def fit(self, X, y):
        self.classes_      = np.unique(y)
        n_samples, n_feats = X.shape
        K = len(self.classes_)

        self._mean      = np.zeros((K, n_feats))
        self._var       = np.zeros((K, n_feats))
        self._log_prior = np.zeros(K)

        for idx, c in enumerate(self.classes_):
            Xc = X[y == c]
            self._mean[idx]      = Xc.mean(axis=0)
            self._var[idx]       = Xc.var(axis=0) + 1e-9   # smoothing
            self._log_prior[idx] = np.log(len(Xc) / n_samples)

        return self

    def _log_likelihood(self, k, x):
        """Log of Gaussian PDF for class k over all features."""
        mu, var = self._mean[k], self._var[k]
        # log N(x; mu, var) = -0.5*log(2*pi*var) - (x-mu)^2 / (2*var)
        return np.sum(-0.5 * np.log(2 * np.pi * var)
                      - (x - mu) ** 2 / (2 * var))

    def predict(self, X):
        preds = []
        for x in X:
            log_post = [
                self._log_prior[k] + self._log_likelihood(k, x)
                for k in range(len(self.classes_))
            ]
            preds.append(self.classes_[np.argmax(log_post)])
        return np.array(preds)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


print('GaussianNaiveBayes class defined successfully.')
print()
print('Methods:')
print('  .fit(X_train, y_train)  — estimate parameters')
print('  .predict(X_test)        — return predicted labels')
print('  .score(X_test, y_test)  — return accuracy')

GaussianNaiveBayes class defined successfully.

Methods:
  .fit(X_train, y_train)  — estimate parameters
  .predict(X_test)        — return predicted labels
  .score(X_test, y_test)  — return accuracy


In [3]:
# Fit the model
gnb = GaussianNaiveBayes()
gnb.fit(X_train, y_train)

# Learned parameters
print('Learned class priors:')
for i, c in enumerate(gnb.classes_):
    print(f'  P({class_names[c]}) = {np.exp(gnb._log_prior[i]):.4f}')

print()
print('Per-class feature means (sepal_l | sepal_w | petal_l | petal_w):')
for i, c in enumerate(gnb.classes_):
    vals = '  '.join(f'{v:.3f}' for v in gnb._mean[i])
    print(f'  {class_names[c]:<12}: {vals}')

# Predict on test set
y_pred = gnb.predict(X_test)

print()
print(f'{"#":<5} {"Actual":<14} {"Predicted":<14} {"Result"}')
print('\u2500' * 46)
for i, (a, p) in enumerate(zip(y_test, y_pred)):
    tag = 'ok' if a == p else 'WRONG'
    print(f'{i:<5} {class_names[a]:<14} {class_names[p]:<14} {tag}')

Learned class priors:
  P(setosa)     = 0.3333
  P(versicolor) = 0.3333
  P(virginica)  = 0.3333

Per-class feature means (sepal_l | sepal_w | petal_l | petal_w):
  setosa      : 5.006  3.428  1.462  0.246
  versicolor  : 5.936  2.770  4.260  1.326
  virginica   : 6.588  2.974  5.552  2.026

#     Actual         Predicted      Result
──────────────────────────────────────────────
0     setosa         setosa         ok
1     setosa         setosa         ok
2     versicolor     versicolor     ok
3     virginica      virginica      ok
4     setosa         setosa         ok
5     versicolor     versicolor     ok
6     versicolor     versicolor     ok
7     virginica      virginica      ok
8     setosa         setosa         ok
9     versicolor     virginica      WRONG
10    setosa         setosa         ok
11    virginica      virginica      ok
12    versicolor     versicolor     ok
13    versicolor     versicolor     ok
14    virginica      virginica      ok
15    setosa         setosa  

In [4]:
# Confusion matrix — computed with numpy (no sklearn)
n_classes = len(gnb.classes_)
cm = np.zeros((n_classes, n_classes), dtype=int)
for a, p in zip(y_test, y_pred):
    cm[a][p] += 1

print('Confusion Matrix  (rows = actual, cols = predicted)')
print()
print(f'{"":>14}', end='')
for name in class_names:
    print(f'  {name:>12}', end='')
print()
print('\u2500' * 56)
for i, name in enumerate(class_names):
    print(f'{name:>13} |', end='')
    for j in range(n_classes):
        print(f'  {cm[i][j]:>12}', end='')
    print()

Confusion Matrix  (rows = actual, cols = predicted)

                       setosa   versicolor    virginica
────────────────────────────────────────────────────────
       setosa |            10            0            0
   versicolor |             0            9            1
    virginica |             0            1            9


In [5]:
def compute_metrics(cm, class_names):
    """
    Compute per-class precision, recall, F1-score from a confusion matrix.
    No sklearn — pure numpy arithmetic.
    """
    n = len(class_names)
    precision, recall, f1 = [], [], []
    for i in range(n):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f = 2*p*r / (p+r)  if (p + r)  > 0 else 0.0
        precision.append(p)
        recall.append(r)
        f1.append(f)
    return np.array(precision), np.array(recall), np.array(f1)

precision, recall, f1 = compute_metrics(cm, class_names)
accuracy = np.trace(cm) / cm.sum()

print(f'Overall Accuracy : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'Correct          : {np.trace(cm)} / {cm.sum()} samples')
print()
print(f'{"Class":<14}  {"Precision":>10}  {"Recall":>10}  {"F1-Score":>10}')
print('\u2500' * 50)
for i, name in enumerate(class_names):
    print(f'{name:<14}  {precision[i]:>10.4f}  {recall[i]:>10.4f}  {f1[i]:>10.4f}')
print('\u2500' * 50)
print(f'{"macro avg":<14}  {precision.mean():>10.4f}  {recall.mean():>10.4f}  {f1.mean():>10.4f}')

Overall Accuracy : 0.9333  (93.33%)
Correct          : 28 / 30 samples

Class             Precision      Recall    F1-Score
──────────────────────────────────────────────────
setosa               1.0000      1.0000      1.0000
versicolor           0.9000      0.9000      0.9000
virginica            0.9000      0.9000      0.9000
──────────────────────────────────────────────────
macro avg            0.9333      0.9333      0.9333
